In [ ]:
from Cryptodome.Cipher import AES
from Cryptodome.Random import get_random_bytes
from Cryptodome.Hash import SHA256
import base64


# -----------------------------------------
# PKCS7 Padding Function
# -----------------------------------------
def pad(data):
    pad_len = 16 - (len(data) % 16)
    return data + bytes([pad_len] * pad_len)


# -----------------------------------------
# Generate AES-256 Key
# -----------------------------------------
def generate_key():
    return get_random_bytes(32)   # 32 bytes = 256 bits


# -----------------------------------------
# Encrypt Message using AES-256-CBC
# -----------------------------------------
def encrypt_message(message, key):

    # Generate random 16-byte IV
    iv = get_random_bytes(16)

    # Create AES-256-CBC cipher
    cipher = AES.new(
        key,
        AES.MODE_CBC,
        iv
    )

    # Convert message into bytes
    message_bytes = message.encode("utf-8")

    # Apply PKCS7 padding
    padded_message = pad(message_bytes)

    # Encrypt the message
    ciphertext = cipher.encrypt(
        padded_message
    )

    # Generate SHA-256 digest
    sha256_hash = SHA256.new(
        ciphertext
    ).digest()

    # Combine IV + Ciphertext
    encrypted_data = iv + ciphertext

    # Convert to Base64
    encrypted_message = base64.b64encode(
        encrypted_data
    ).decode("utf-8")

    return (
        encrypted_message,
        key,
        iv,
        ciphertext,
        sha256_hash
    )


# -----------------------------------------
# MAIN - SENDER SIDE
# -----------------------------------------
if __name__ == "__main__":

    print("==========================================")
    print("       AI-ASSISTED SECURE MESSAGING")
    print("              SENDER SIDE")
    print("==========================================")

    # Take message from sender
    message = input("\nEnter message: ")

    # Generate AES-256 key
    key = generate_key()

    # Encrypt message
    (
        encrypted_message,
        key,
        iv,
        ciphertext,
        sha256_hash
    ) = encrypt_message(
        message,
        key
    )

    print("\n------------------------------------------")

    print("AES-256 KEY:")
    print(key.hex())

    print("\nRANDOM IV:")
    print(iv.hex())

    print("\nCIPHERTEXT:")
    print(ciphertext.hex())

    print("\nSHA-256 DIGEST:")
    print(sha256_hash.hex())

    print("\nENCRYPTED MESSAGE (Base64):")
    print(encrypted_message)

    print("\n------------------------------------------")
    print("Message encrypted successfully.")
    print("Ready for secure transmission.")
    print("------------------------------------------")

       AI-ASSISTED SECURE MESSAGING
              SENDER SIDE

Enter message: bye bye

------------------------------------------
AES-256 KEY:
9bb608249ccf4e1e1ffc48f9aa875182c33067430fbf06a88bb4f43343263cbb

RANDOM IV:
62f18666190591cd6b5b243f9fade769

CIPHERTEXT:
0bb16de2edc29a1c9f3a12e9431b6d41

SHA-256 DIGEST:
994f859a3ffe5f9773638794983ee32388b6667bd79cdf07f8a4996dd5d334ad

ENCRYPTED MESSAGE (Base64):
YvGGZhkFkc1rWyQ/n63naQuxbeLtwpocnzoS6UMbbUE=

------------------------------------------
Message encrypted successfully.
Ready for secure transmission.
------------------------------------------


In [ ]:
from Cryptodome.Cipher import AES
from Cryptodome.Hash import SHA256
import base64


# -----------------------------------------
# PKCS7 Unpadding Function
# -----------------------------------------
def unpad(data):
    pad_len = data[-1]

    if pad_len < 1 or pad_len > 16:
        raise ValueError("Invalid padding")

    if data[-pad_len:] != bytes([pad_len]) * pad_len:
        raise ValueError("Invalid padding")

    return data[:-pad_len]


# -----------------------------------------
# SHA-256 Verification
# -----------------------------------------
def verify_sha256(ciphertext, received_hash):
    calculated_hash = SHA256.new(ciphertext).digest()

    return calculated_hash == received_hash


# -----------------------------------------
# AES-256-CBC Decryption
# -----------------------------------------
def decrypt_message(encrypted_message, key, received_hash):

    # Convert Base64 string back to bytes
    encrypted_data = base64.b64decode(encrypted_message)

    # -------------------------------------
    # Extract IV
    # -------------------------------------
    iv = encrypted_data[:16]

    # -------------------------------------
    # Extract ciphertext
    # -------------------------------------
    ciphertext = encrypted_data[16:]

    # -------------------------------------
    # Verify SHA-256 integrity
    # -------------------------------------
    if not verify_sha256(ciphertext, received_hash):
        print("\nWARNING!")
        print("Message integrity verification FAILED.")
        print("The message may have been modified or corrupted.")

        return None

    print("\nIntegrity verification successful.")

    # -------------------------------------
    # AES-256-CBC Decryption
    # -------------------------------------
    cipher = AES.new(
        key,
        AES.MODE_CBC,
        iv
    )

    padded_plaintext = cipher.decrypt(ciphertext)

    # -------------------------------------
    # Remove PKCS7 padding
    # -------------------------------------
    plaintext = unpad(padded_plaintext)

    # Convert bytes to text
    return plaintext.decode("utf-8")


# -----------------------------------------
# MAIN - RECEIVER SIDE
# -----------------------------------------
if __name__ == "__main__":

    print("==========================================")
    print("       AI-ASSISTED SECURE MESSAGING")
    print("             RECEIVER SIDE")
    print("==========================================")

    # -------------------------------------
    # Receive encrypted message
    # -------------------------------------
    encrypted_message = input(
        "\nEnter encrypted message (Base64): "
    )

    # -------------------------------------
    # Receive AES-256 key
    # -------------------------------------
    key_hex = input(
        "\nEnter AES-256 key (hex): "
    )

    # Convert hexadecimal key to bytes
    key = bytes.fromhex(key_hex)

    # Check AES-256 key size
    if len(key) != 32:
        print("\nERROR!")
        print("AES-256 key must be exactly 32 bytes.")
        exit()

    # -------------------------------------
    # Receive SHA-256 digest
    # -------------------------------------
    hash_hex = input(
        "\nEnter SHA-256 digest (hex): "
    )

    received_hash = bytes.fromhex(hash_hex)

    # Check SHA-256 digest size
    if len(received_hash) != 32:
        print("\nERROR!")
        print("SHA-256 digest must be 32 bytes.")
        exit()

    # -------------------------------------
    # Decrypt message
    # -------------------------------------
    try:

        plaintext = decrypt_message(
            encrypted_message,
            key,
            received_hash
        )

        if plaintext is not None:

            print("\n------------------------------------------")
            print("Decryption successful!")
            print("------------------------------------------")

            print("\nOriginal Message:")
            print(plaintext)

            print("\n------------------------------------------")
            print("Message received successfully.")
            print("------------------------------------------")

    except Exception as e:

        print("\nERROR!")
        print("Decryption failed.")
        print("Reason:", e)

       AI-ASSISTED SECURE MESSAGING
             RECEIVER SIDE

Enter encrypted message (Base64): YvGGZhkFkc1rWyQ/n63naQuxbeLtwpocnzoS6UMbbUE=

Enter AES-256 key (hex): 9bb608249ccf4e1e1ffc48f9aa875182c33067430fbf06a88bb4f43343263cbb

Enter SHA-256 digest (hex): 994f859a3ffe5f9773638794983ee32388b6667bd79cdf07f8a4996dd5d334ad

Integrity verification successful.

------------------------------------------
Decryption successful!
------------------------------------------

Original Message:
bye bye

------------------------------------------
Message received successfully.
------------------------------------------
